<a href="https://colab.research.google.com/github/TarfaMajeed/Projects/blob/main/Text_Emotion_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1: Install & Login
%%capture
!pip install -q transformers sentence-transformers faiss-cpu scikit-learn pandas openpyxl tabulate

print("✅ Packages Installed")

# Hugging Face Login (Removes Warning + Faster Download)
from huggingface_hub import login
from google.colab import userdata

try:
    login(token=userdata.get('HF_TOKEN'))
    print("✅ Logged into Hugging Face")
except:
    print("⚠️  No HF_TOKEN found in secrets. You can continue but it may be slower.")

In [ ]:
# CELL 2: Imports + Mount Drive + Load Data
from google.colab import drive
drive.mount('/content/drive')

import os, warnings, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tabulate import tabulate

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {DEVICE} (Make sure T4 GPU is selected)')

In [ ]:
# CELL 2.1: Load Dataset
EXCEL_PATH = '/content/drive/MyDrive/text_emotion_classifier/RU-EN-Emotion Dataset.xlsx'

df = pd.read_excel(EXCEL_PATH, sheet_name='Annotation Dataset')
df = df[['Tweets', 'Level 2']].copy()
df.columns = ['text', 'label']
df = df.dropna().reset_index(drop=True)
df['text'] = df['text'].astype(str).str.strip()
df['label'] = df['label'].astype(str).str.strip().str.title()

le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])

train_df, temp_df = train_test_split(df, test_size=0.25, stratify=df['label_enc'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label_enc'], random_state=42)

print(f"Total Samples: {len(df)}")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("\nClasses:", list(le.classes_))

In [ ]:
# CELL 3: XLM-R Transformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

TR_MODEL = 'xlm-roberta-base'
TR_MAX_LEN = 64
TR_BS = 32
TR_EPOCHS = 10
TR_LR = 2e-5

tok_tr = AutoTokenizer.from_pretrained(TR_MODEL)

class EmotionDS(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.enc = tokenizer(list(texts), truncation=True, padding='max_length',
                            max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()} | {'labels': self.labels[i]}

train_ds = EmotionDS(train_df['text'], train_df['label_enc'], tok_tr, TR_MAX_LEN)
val_ds   = EmotionDS(val_df['text'],   val_df['label_enc'],   tok_tr, TR_MAX_LEN)
test_ds  = EmotionDS(test_df['text'],  test_df['label_enc'],  tok_tr, TR_MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=TR_BS, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

model_tr = AutoModelForSequenceClassification.from_pretrained(
    TR_MODEL, num_labels=len(le.classes_), ignore_mismatched_sizes=True
).to(DEVICE)

optimizer = AdamW(model_tr.parameters(), lr=TR_LR, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=='cuda'))

def train_epoch(model, loader, optimizer=None, train=True):
    model.train() if train else model.eval()
    total_loss, preds, labels = 0, [], []
    with torch.set_grad_enabled(train):
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
                out = model(**batch)
                loss = out.loss
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item()
            preds.extend(out.logits.argmax(-1).cpu().numpy())
            labels.extend(batch['labels'].cpu().numpy())
    return total_loss / len(loader), accuracy_score(labels, preds)

print("🚀 Training XLM-R (10 Epochs)... This will take the most time.")
best_acc = 0
for epoch in range(TR_EPOCHS):
    tr_loss, tr_acc = train_epoch(model_tr, train_loader, optimizer, train=True)
    vl_loss, vl_acc = train_epoch(model_tr, val_loader, train=False)
    print(f"Epoch {epoch+1:2d} | Train: {tr_acc:.4f} | Val: {vl_acc:.4f}")
    if vl_acc > best_acc:
        best_acc = vl_acc
        best_state = {k: v.cpu().clone() for k, v in model_tr.state_dict().items()}

model_tr.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
_, TR_ACC = train_epoch(model_tr, test_loader, train=False)
print(f"\n✅ XLM-R Test Accuracy: {TR_ACC:.4f}")

In [ ]:
# CELL 4: LLM - DistilBERT (Fast & Good Performance)
from transformers import AutoTokenizer, AutoModelForSequenceClassification

LLM_MODEL = 'distilbert-base-multilingual-cased'
LLM_MAX_LEN = 64
LLM_BS = 48
LLM_EPOCHS = 6

tok_llm = AutoTokenizer.from_pretrained(LLM_MODEL)

# Reuse the same EmotionDS class from previous cell
llm_train_ds = EmotionDS(train_df['text'], train_df['label_enc'], tok_llm, LLM_MAX_LEN)
llm_val_ds   = EmotionDS(val_df['text'],   val_df['label_enc'],   tok_llm, LLM_MAX_LEN)
llm_test_ds  = EmotionDS(test_df['text'],  test_df['label_enc'],  tok_llm, LLM_MAX_LEN)

llm_train_loader = DataLoader(llm_train_ds, batch_size=LLM_BS, shuffle=True, num_workers=2, pin_memory=True)
llm_val_loader   = DataLoader(llm_val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
llm_test_loader  = DataLoader(llm_test_ds,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

model_llm = AutoModelForSequenceClassification.from_pretrained(
    LLM_MODEL, num_labels=len(le.classes_), ignore_mismatched_sizes=True
).to(DEVICE)

optimizer_llm = AdamW(model_llm.parameters(), lr=3e-5)

print("🚀 Training DistilBERT (LLM) - 6 Epochs...")
best_acc_llm = 0
for epoch in range(LLM_EPOCHS):
    tr_loss, tr_acc = train_epoch(model_llm, llm_train_loader, optimizer_llm, train=True)
    vl_loss, vl_acc = train_epoch(model_llm, llm_val_loader, train=False)
    print(f"Epoch {epoch+1:2d} | Train: {tr_acc:.4f} | Val: {vl_acc:.4f}")
    if vl_acc > best_acc_llm:
        best_acc_llm = vl_acc
        best_llm_state = {k: v.cpu().clone() for k, v in model_llm.state_dict().items()}

model_llm.load_state_dict({k: v.to(DEVICE) for k, v in best_llm_state.items()})
_, LLM_ACC = train_epoch(model_llm, llm_test_loader, train=False)
print(f"\n✅ DistilBERT (LLM) Test Accuracy: {LLM_ACC:.4f}")

In [ ]:
# CELL 5: LSTM Model
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import re

def tokenize(text):
    return re.findall(r'\w+', text.lower())

# Create Vocabulary
all_tokens = [token for text in train_df['text'] for token in tokenize(text)]
vocab = {word: idx+1 for idx, (word, _) in enumerate(Counter(all_tokens).most_common(15000))}
vocab['<PAD>'] = 0

def text_to_seq(text):
    return [vocab.get(word, 0) for word in tokenize(text)]

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_classes=len(le.classes_)):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim*2, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        out = torch.mean(out, dim=1)
        return self.fc(out)

# Prepare sequences
train_seq = [text_to_seq(t) for t in train_df['text']]
val_seq   = [text_to_seq(t) for t in val_df['text']]
test_seq  = [text_to_seq(t) for t in test_df['text']]

def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded = pad_sequence([torch.tensor(t) for t in texts], batch_first=True, padding_value=0)
    return texts_padded, torch.tensor(labels)

train_lstm_ds = list(zip(train_seq, train_df['label_enc'].values))
val_lstm_ds   = list(zip(val_seq,   val_df['label_enc'].values))
test_lstm_ds  = list(zip(test_seq,  test_df['label_enc'].values))

train_lstm_dl = DataLoader(train_lstm_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_lstm_dl   = DataLoader(val_lstm_ds,   batch_size=128, shuffle=False, collate_fn=collate_fn)
test_lstm_dl  = DataLoader(test_lstm_ds,  batch_size=128, shuffle=False, collate_fn=collate_fn)

# Train LSTM
model_lstm = LSTMModel(len(vocab)).to(DEVICE)
optimizer_lstm = AdamW(model_lstm.parameters(), lr=1e-3)

print("🚀 Training LSTM (8 Epochs)...")
best_acc_lstm = 0
for epoch in range(8):
    model_lstm.train()
    for x, y in train_lstm_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer_lstm.zero_grad()
        loss = nn.CrossEntropyLoss()(model_lstm(x), y)
        loss.backward()
        optimizer_lstm.step()

    # Validation
    model_lstm.eval()
    preds, labs = [], []
    with torch.no_grad():
        for x, y in val_lstm_dl:
            x = x.to(DEVICE)
            preds.extend(model_lstm(x).argmax(1).cpu().numpy())
            labs.extend(y.numpy())
    acc = accuracy_score(labs, preds)
    print(f"LSTM Epoch {epoch+1:2d} | Val Acc: {acc:.4f}")
    if acc > best_acc_lstm:
        best_acc_lstm = acc
        torch.save(model_lstm.state_dict(), 'best_lstm.pth')

model_lstm.load_state_dict(torch.load('best_lstm.pth'))
model_lstm.eval()

# Test Accuracy
lstm_preds = []
with torch.no_grad():
    for x, _ in test_lstm_dl:
        x = x.to(DEVICE)
        lstm_preds.extend(model_lstm(x).argmax(1).cpu().numpy())

LSTM_ACC = accuracy_score(test_df['label_enc'], lstm_preds)
print(f"\n✅ LSTM Test Accuracy: {LSTM_ACC:.4f}")

In [ ]:
# CELL 6: RAG Model (No Training Needed)
from sentence_transformers import SentenceTransformer
import faiss

st_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("Encoding training sentences for RAG...")
train_embs = st_model.encode(train_df['text'].tolist(), batch_size=512, show_progress_bar=True, normalize_embeddings=True)

index = faiss.IndexFlatIP(train_embs.shape[1])
index.add(train_embs)
rag_labels = train_df['label_enc'].values

def rag_predict(texts, k=5):
    embs = st_model.encode(texts, batch_size=512, normalize_embeddings=True)
    _, I = index.search(embs, k)
    preds = [np.bincount(rag_labels[idx]).argmax() for idx in I]
    return np.array(preds)

rag_preds = rag_predict(test_df['text'].tolist())
RAG_ACC = accuracy_score(test_df['label_enc'], rag_preds)
print(f"✅ RAG (FAISS) Test Accuracy: {RAG_ACC:.4f}")

In [ ]:
# CELL 7: Results Comparison
results_df = pd.DataFrame({
    'Model': ['Transformer (XLM-R)', 'LLM (DistilBERT)', 'LSTM', 'RAG (FAISS)'],
    'Test Accuracy': [TR_ACC, LLM_ACC, LSTM_ACC, RAG_ACC]
})

print("\n" + "="*75)
print("🎉 FINAL RESULTS - ROMAN URDU EMOTION CLASSIFIER")
print("="*75)
print(tabulate(results_df.round(4), headers='keys', tablefmt='fancy_grid', showindex=False))

In [ ]:
# CELL 8: Interactive Emotion Detector
model_tr.eval()
model_llm.eval()
model_lstm.eval()

def predict(text, model_choice='1'):
    if model_choice == '1':      # XLM-R
        enc = tok_tr([text], truncation=True, padding='max_length', max_length=TR_MAX_LEN, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            logits = model_tr(**enc).logits
        pred_id = logits.argmax(-1).item()
        conf = torch.softmax(logits, -1).max().item()
        label = le.inverse_transform([pred_id])[0]
        return label, f"{conf:.1%}"

    elif model_choice == '2':    # DistilBERT
        enc = tok_llm([text], truncation=True, padding='max_length', max_length=LLM_MAX_LEN, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            logits = model_llm(**enc).logits
        pred_id = logits.argmax(-1).item()
        conf = torch.softmax(logits, -1).max().item()
        label = le.inverse_transform([pred_id])[0]
        return label, f"{conf:.1%}"

    elif model_choice == '3':    # LSTM
        seq = torch.tensor([text_to_seq(text)]).to(DEVICE)
        with torch.no_grad():
            logits = model_lstm(seq)
        pred_id = logits.argmax(-1).item()
        label = le.inverse_transform([pred_id])[0]
        return label, "LSTM"

    elif model_choice == '4':    # RAG
        pred_id = rag_predict([text])[0]
        label = le.inverse_transform([pred_id])[0]
        return label, "RAG (kNN)"

print("\n" + "="*75)
print("🎭 INTERACTIVE EMOTION DETECTOR")
print("="*75)
print("Available Models:")
print("1 → Transformer (XLM-R)")
print("2 → LLM (DistilBERT)")
print("3 → LSTM")
print("4 → RAG (FAISS)")

while True:
    txt = input("\nEnter Roman Urdu text (or 'quit'): ").strip()
    if txt.lower() in ['quit', 'exit', 'q']:
        print("👋 Goodbye!")
        break
    if not txt:
        continue

    choice = input("Choose Model (1-4): ").strip()
    if choice not in ['1','2','3','4']:
        choice = '1'
        print("Defaulted to Model 1 (XLM-R)")

    label, conf = predict(txt, choice)
    print(f"🎯 Predicted Emotion: **{label.upper()}**  ({conf})")